# Step 5: Fine-tune ReactionT5 on ORD (Model 1, Colab GPU)

Runs `scripts/train_reactant_model_ord.py`, which fine-tunes the ORD-pretrained
checkpoint `sagawa/ReactionT5v2-retrosynthesis` (before any USPTO-specific
fine-tuning) on the freshly built, leak-checked `data/v2_ord_train/reactants_train.jsonl`.

**Colab session budget: ~3h/day.** Checkpoints are written to Google Drive, and this
notebook can simply be re-run on a later day -- it auto-resumes from the last
checkpoint. Do not clear the Drive folder between sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**Reclaim Drive quota (optional, run once per session):** the training script no
longer rotates/deletes checkpoints directly on Drive at all (see the v5 note below --
Trainer now checkpoints on local Colab disk and only ever *overwrites* one fixed
Drive folder), so this matters much less than before. Still useful once, to clear
out anything trashed by earlier (v2-v4) runs before this fix. First run prompts an
auth popup.

**Warning:** this empties Trash for your **entire** Google Drive account, not just
this project's files -- anything else you'd trashed elsewhere and might still want
to recover will be gone permanently too. Skip this cell if that matters to you.

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
build("drive", "v3").files().emptyTrash().execute()
print("Drive Trash emptied.")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

`data/v2_ord_train/` and `data/v2_uspto_train/` are gitignored (large, derived) --
regenerate them deterministically here (fixed seed, excludes the committed
`data/v2_ord_eval_targets.json`/`data/v2_uspto_eval_targets.json` by construction,
so this can never leak into the eval sets). Only needs to run once per Colab
session; skipped automatically if the files already exist (e.g. this is a
same-day resume).

In [ ]:
import os

if not os.path.exists("data/v2_ord_train/reactants_train.jsonl"):
    !python scripts/build_train_data_ord.py --pool-count 60000 --seed 42
if not os.path.exists("data/v2_uspto_train/reactants_train.jsonl"):
    !python scripts/build_train_data_uspto.py

**v2 result (for reference):** ORD-only data + lr=5e-5 + warmup + best-checkpoint
selection recovered from an earlier degraded run and gave a real improvement over
the pre-fine-tune baseline: exact_match 43.7%→50.7%, core_exact_match 54.3%→62.3%
on the 300-target ORD eval set (USPTO also improved slightly, 16.3%→21.7%).

**v3 result:** added always-on SMILES augmentation, but overfit fast (eval_loss rose
monotonically from the first measurement). **v4** lowered the learning rate further,
added weight decay, and made augmentation partial (prob=0.5) -- eval_loss was
improving smoothly and had *not* turned upward by the time the run was interrupted
(a good sign), but v4's Drive folder ended up in an inconsistent state: Trainer was
rotating checkpoints directly on the Drive-mounted path, Drive's FUSE mount moves
deletes to Trash instead of freeing them, and after a manual Trash-restore some
checkpoint folders came back only partially (empty shells) -- the resume logic
couldn't reliably tell which checkpoint was real anymore, and training restarted
from scratch instead of continuing.

**v5 (this run): same hyperparameters as v4, fixed checkpoint architecture.**
Trainer now checkpoints and rotates on **local Colab disk**
(`--local-work-dir`, default `/content/local_model1_work`) instead of directly on
Drive -- fast, reliable, ordinary filesystem behavior, no Trash involved. After
every save, a callback copies the checkpoint out to
`{output_dir}/latest_checkpoint` on Drive, always overwriting the same filenames in
place (never deleting anything there). Resuming reads only that one folder. This
also fixes a related bug: the previous session's `best_model_checkpoint` reference
pointed at a local path that no longer exists after a session restart -- the script
now detects and repoints it automatically. `output_dir` below points at a *new*
`model1_reactant_v5` folder (the old `model1_reactant_v4` folder's confused state
can be deleted once this is confirmed working).

In [ ]:
output_dir = "/content/drive/MyDrive/retro-planner-checkpoints/model1_reactant_v5"  # @param {type:"string"}
time_budget_minutes = 165  # @param {type:"number"}
mix_in_uspto = False  # @param {type:"boolean"}

extra_flag = ["--extra-train-file", "data/v2_uspto_train/reactants_train.jsonl"] if mix_in_uspto else []

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!python scripts/train_reactant_model_ord.py \
    --output-dir "{output_dir}" \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(extra_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Training output is redirected to `train.log` in `output_dir` (on Drive) instead of
printing in this cell -- a long run's per-step tqdm bar and log lines used to grow the
notebook's output DOM large enough to make the browser tab unresponsive after 1-2 hours,
even though the actual training was proceeding fine underneath. The cell above still
blocks until training stops (so Colab doesn't treat the runtime as idle), but prints
nothing while it runs. To check progress without waiting: open `train.log` directly in
Google Drive's own web preview (refresh it there) -- that works independently of the
Colab kernel, which stays busy running the cell above.

Re-run the cell above (same `output_dir`) on the next day's Colab session to continue
training -- it reads `{output_dir}/latest_checkpoint` (the one Drive checkpoint the
sync callback keeps up to date) and resumes from there; local Colab disk being wiped
between sessions no longer matters. Once `trainer.train()` finishes (not just
time-budget-stopped), the final model is also saved to `{output_dir}/final`.